# 04 - Diagnostics and figures  (Stage 4)

Pathway maps, where each group is at each radius, how long each group takes to
get there, seasonal / interannual fractions, centroids, and the most typical
particles per label.

`LABEL_COL = "cluster_group"` for the merged groups (needs `config.GROUP_MAP`)
or `"cluster_label"` for the raw clusters.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)
print(f"crossing: {C.LAT_CROSS} N | radii (km): {C.RADII_KM}")

import joblib
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cfeature
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap

In [ ]:
BEST_K = 70                    # must match notebook 03 (papermill: -p BEST_K <k>)
LABEL_COL = "cluster_group"    # "cluster_group" (merged) or "cluster_label" (raw)

In [ ]:
lab = pd.read_parquet(C.LABELED_FILE)
MODEL_DIR = C.manual_model_dir(BEST_K)
km     = joblib.load(MODEL_DIR / f"kmeans_k{BEST_K}.pkl")
scaler = joblib.load(MODEL_DIR / "scaler.pkl")
W = P.feature_weight_vector()

labelled = lab[lab[LABEL_COL] >= 0].copy()
n_lab = int(labelled[LABEL_COL].max()) + 1
cmap = plt.get_cmap("tab20", max(n_lab, 3))
LABEL_NAMES = C.GROUP_NAMES if LABEL_COL == "cluster_group" else {}

def label_name(c):
    return LABEL_NAMES.get(c, str(c))

# centroids in degrees; group centroid = size-weighted mean in the fitted space
if LABEL_COL == "cluster_group" and C.GROUP_MAP:
    _, raw2grp = P.apply_group_map(np.arange(BEST_K), C.GROUP_MAP, BEST_K)
    sizes = lab.loc[lab.cluster_label >= 0, "cluster_label"].value_counts()
    centers = np.zeros((n_lab, km.cluster_centers_.shape[1])); wsum = np.zeros(n_lab)
    for c in range(BEST_K):
        g, w = raw2grp[c], float(sizes.get(c, 0))
        centers[g] += km.cluster_centers_[c] * w; wsum[g] += w
    centers /= np.where(wsum[:, None] == 0, 1, wsum[:, None])
else:
    centers = km.cluster_centers_[:n_lab]
disp_cent = P.centroids_to_degrees(centers / W, scaler)
start_lon = labelled.groupby(LABEL_COL).cross_lon.mean().reindex(range(n_lab)).to_numpy()

pc = ccrs.PlateCarree()
EXT = [C.DOMAIN["lon_min"], C.DOMAIN["lon_max"], C.DOMAIN["lat_min"], C.DOMAIN["lat_max"]]
def geo_axes_grid(n, w=4.2, h=3.4, ncol=4):
    ncol = min(n, ncol); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(w * ncol, h * nrow),
                             subplot_kw={"projection": pc}, squeeze=False)
    axf = axes.ravel()
    for a in axf[n:]:
        a.axis("off")
    for a in axf[:n]:
        a.set_extent(EXT, crs=pc)
    return fig, axf
def land(ax):
    ax.add_feature(cfeature.LAND, facecolor="0.85", zorder=2); ax.coastlines(lw=.5, zorder=3)
print(f"labelled particles: {len(labelled):,} | {LABEL_COL}: {n_lab} labels")

## Figure 1 - Pathway footprint

Up to `C.PLOT_SAMPLE_PER_LABEL` particles per label are read from the Zarr
stores (by `store` / `row`, checked against `traj`). With `FROM_CROSSING =
True` only the part after the 2.5N crossing is binned - the part the clustering
saw; `False` bins the whole track from release (obs 0 skipped).

In [ ]:
FROM_CROSSING = True
N_IO_THREADS  = int(os.environ.get("SLURM_CPUS_PER_TASK", 16))

n_plot = C.PLOT_SAMPLE_PER_LABEL
pick = (labelled.groupby(LABEL_COL, group_keys=False)
        .apply(lambda d: d.sample(min(n_plot, len(d)), random_state=C.RANDOM_STATE)))
pick = pick[["trajectory_id", "store", "row", "traj", "cross_obs", LABEL_COL]] \
           .rename(columns={LABEL_COL: "lab"})
paths = P.read_paths(pick, n_jobs=N_IO_THREADS)
print(f"read {len(paths):,} trajectories from {pick.store.nunique()} stores")

lon_bins = np.linspace(C.DOMAIN["lon_min"], C.DOMAIN["lon_max"], 100 * 4)
lat_bins = np.linspace(C.DOMAIN["lat_min"], C.DOMAIN["lat_max"], 35 * 4)
xs = {c: [] for c in range(n_lab)}; ys = {c: [] for c in range(n_lab)}
for tid, c, k0 in zip(pick.trajectory_id, pick.lab, pick.cross_obs):
    lo, la = paths[int(tid)]
    s = k0 if FROM_CROSSING else 1
    xs[c].append(lo[s:]); ys[c].append(la[s:])
Hlab = np.zeros((n_lab, lat_bins.size - 1, lon_bins.size - 1))
for c in range(n_lab):
    if not xs[c]:
        continue
    x = np.concatenate(xs[c]); y = np.concatenate(ys[c])
    m = ~(np.isnan(x) | np.isnan(y))
    Hlab[c] = np.histogram2d(x[m], y[m], bins=[lon_bins, lat_bins])[0].T

fig = plt.figure(figsize=(11, 7))
ax = plt.axes(projection=pc); ax.set_extent(EXT, crs=pc)
for c in range(n_lab):
    ax.pcolormesh(lon_bins, lat_bins, np.where(Hlab[c] > 0, 1.0, np.nan),
                  cmap=ListedColormap([cmap(c)]), vmin=0, vmax=1, alpha=.5,
                  transform=pc, zorder=1)
land(ax)
ax.legend(handles=[Line2D([0], [0], marker="s", ls="", mec="none", alpha=.5, mfc=cmap(c),
                          label=f"{label_name(c)} (n={int((pick.lab == c).sum())})")
                   for c in range(n_lab)],
          title=LABEL_COL, fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title(f"Pathways by {LABEL_COL} (k={BEST_K}, {n_lab} labels)"
             + (" - after crossing 2.5N" if FROM_CROSSING else ""))
fig.savefig(C.FIG_DIR / f"pathway_map_k{BEST_K}_{LABEL_COL}.png", dpi=140, bbox_inches="tight")
plt.show()

## Figure 1b - one panel per label

In [ ]:
fig, axf = geo_axes_grid(n_lab)
for c in range(n_lab):
    ax = axf[c]
    ax.pcolormesh(lon_bins, lat_bins, np.where(Hlab[c] > 0, 1.0, np.nan),
                  cmap=ListedColormap([cmap(c)]), vmin=0, vmax=1, alpha=.6,
                  transform=pc, zorder=1)
    land(ax)
    ax.set_title(f"{label_name(c)} (n={int((pick.lab == c).sum())})", fontsize=9,
                 color=cmap(c), fontweight="bold")
fig.suptitle(f"Pathways per {LABEL_COL} (k={BEST_K})", y=1.0, fontsize=12)
fig.tight_layout()
fig.savefig(C.FIG_DIR / f"pathway_maps_by_{LABEL_COL}_k{BEST_K}.png", dpi=130, bbox_inches="tight")
plt.show()

## Figure 1c - where each group exits one circle, and timing per radius

The footprint integrates the whole track. The first figure freezes one radius
(`SNAP_R`): where each group's particles leave that circle. The second shows,
per group, the median days after crossing to reach each radius (25-75% band) -
the timing the clustering itself ignores.

In [ ]:
SNAP_R = C.RADII_KM[2]          # any radius in C.RADII_KM

la_c, lo_c = P.lat_col(SNAP_R), P.lon_col(SNAP_R)
snap = labelled[[LABEL_COL, la_c, lo_c]].dropna()
fig, axf = geo_axes_grid(n_lab, h=2.6)
for c in range(n_lab):
    s = snap[snap[LABEL_COL] == c]
    h = np.histogram2d(s[lo_c], s[la_c], bins=[lon_bins, lat_bins])[0]
    axf[c].pcolormesh(lon_bins, lat_bins, np.where(h.T > 0, 1.0, np.nan),
                      cmap=ListedColormap([cmap(c)]), vmin=0, vmax=1, alpha=.7,
                      transform=pc, zorder=1)
    land(axf[c])
    axf[c].set_title(f"{label_name(c)}  (n={len(s):,})", fontsize=9, color=cmap(c),
                     fontweight="bold")
fig.suptitle(f"Where each group leaves the {SNAP_R} km circle (k={BEST_K})", y=1.0)
fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 5.5))
for c in range(n_lab):
    s = labelled[labelled[LABEL_COL] == c]
    q = np.array([s[P.days_col(r)].quantile([.25, .5, .75]).to_numpy() for r in C.RADII_KM])
    ax.fill_between(C.RADII_KM, q[:, 0], q[:, 2], color=cmap(c), alpha=.12, lw=0)
    ax.plot(C.RADII_KM, q[:, 1], "-o", color=cmap(c), lw=1.8, ms=4, label=label_name(c))
ax.set_xlabel("radius around the crossing point [km]")
ax.set_ylabel("days after crossing to get there")
ax.set_xticks(C.RADII_KM); ax.grid(axis="y", color="0.9", lw=.6)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
ax.legend(fontsize=8, ncol=2, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.set_title("Time to reach each radius per group (median, 25-75%; particles that "
             "got there)", fontsize=10, loc="left")
fig.tight_layout(); plt.show()

## Figure 2 - Seasonal fractions (by crossing month and by release month)

In [ ]:
def fraction_table(group_col):
    g = labelled.groupby([group_col, LABEL_COL]).size().unstack(fill_value=0)
    return g.div(g.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, col in zip(axes, ["cross_month", "release_month"]):
    seas = fraction_table(col).reindex(range(1, 13)).fillna(0)
    bottom = np.zeros(12)
    for c in range(n_lab):
        v = seas.get(c, pd.Series(0, index=seas.index)).to_numpy()
        ax.bar(seas.index, v, bottom=bottom, color=cmap(c), label=label_name(c))
        bottom += v
    ax.set_xlabel(col.replace("_", " ")); ax.set_xticks(range(1, 13))
axes[0].set_ylabel("% of particles")
axes[1].legend(ncol=2, fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
fig.suptitle(f"Seasonal fractions ({LABEL_COL})")
fig.tight_layout()
fig.savefig(C.FIG_DIR / f"seasonal_fractions_k{BEST_K}_{LABEL_COL}.png", dpi=130, bbox_inches="tight")
plt.show()

## Figure 3 - Interannual fractions (by crossing year)

In [ ]:
yr = fraction_table("cross_year").sort_index()
fig, ax = plt.subplots(figsize=(11, 5))
ax.stackplot(yr.index, *[yr.get(c, pd.Series(0, index=yr.index)).to_numpy() for c in range(n_lab)],
             colors=[cmap(c) for c in range(n_lab)], labels=[label_name(c) for c in range(n_lab)])
ax.set_xlabel("crossing year"); ax.set_ylabel("% of particles"); ax.set_ylim(0, 100)
ax.set_title(f"Interannual fractions ({LABEL_COL})")
ax.legend(ncol=2, fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
fig.savefig(C.FIG_DIR / f"interannual_fractions_k{BEST_K}_{LABEL_COL}.png", dpi=130, bbox_inches="tight")
plt.show()

## Figure 4 - Days to reach `TRANSIT_R` per label
From `days_<R>km` in the features file (no Zarr reading).

In [ ]:
TRANSIT_R = C.RADII_KM[2]
col = P.days_col(TRANSIT_R)
ncol = 5; nrow = int(np.ceil(n_lab / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3 * ncol, 2.4 * nrow), squeeze=False)
for c in range(n_lab):
    ax = axes[c // ncol][c % ncol]
    d = labelled.loc[labelled[LABEL_COL] == c, col].dropna()
    ax.hist(d, bins=40, color=cmap(c))
    ax.set_title(f"{label_name(c)} (median {d.median():.0f} d)", fontsize=8)
    ax.set_xlabel(f"days crossing -> {TRANSIT_R} km", fontsize=7)
for j in range(n_lab, nrow * ncol):
    axes[j // ncol][j % ncol].axis("off")
fig.tight_layout()
fig.savefig(C.FIG_DIR / f"transit_{TRANSIT_R}km_k{BEST_K}_{LABEL_COL}.png", dpi=120)
plt.show()

## Figure 5 - Centroids (mean crossing point -> exit at each radius)

In [ ]:
pct = labelled[LABEL_COL].value_counts(normalize=True).sort_index() * 100
markers = ["o", "s", "*", "^", "D", "P"]
fig = plt.figure(figsize=(11, 7))
ax = plt.axes(projection=pc); ax.set_extent(EXT, crs=pc); land(ax)
ax.axhline(C.LAT_CROSS, color="k", lw=.6, ls="--")
for c in range(n_lab):
    lats = disp_cent[c, 0::2]; lons = disp_cent[c, 1::2]
    ax.plot(np.r_[start_lon[c], lons], np.r_[C.LAT_CROSS, lats], "-", color=cmap(c),
            lw=1, alpha=.6, transform=pc)
    for j in range(len(C.RADII_KM)):
        ax.scatter(lons[j], lats[j], color=cmap(c), s=90, marker=markers[j % len(markers)],
                   edgecolor="k", transform=pc, zorder=4)
    ax.text(lons[-1], lats[-1], f" {label_name(c)} ({pct.get(c, 0):.0f}%)", fontsize=8, transform=pc)
ax.legend(handles=[Line2D([0], [0], marker=markers[j % len(markers)], color="k", ls="",
                          mfc="0.7", label=f"{r} km") for j, r in enumerate(C.RADII_KM)],
          loc="lower right", fontsize=8)
ax.set_title(f"Centroids ({LABEL_COL}, k={BEST_K})")
fig.savefig(C.FIG_DIR / f"centroids_k{BEST_K}_{LABEL_COL}.png", dpi=140, bbox_inches="tight")
plt.show()

## Table 1 - Summary per label

In [ ]:
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
R_last = C.RADII_KM[-1]
rows = []
for c in range(n_lab):
    s = labelled[labelled[LABEL_COL] == c]
    row = dict(label=c, name=label_name(c), pct=round(100 * len(s) / len(labelled), 1),
               pct_partial=round(100 * (s.status == "partial").mean(), 1),
               peak_cross_month=month_names[int(s.cross_month.mode().iloc[0]) - 1],
               median_days_to_cross=round(float(s.days_to_cross.median()), 1))
    for r in C.RADII_KM:
        row[f"median_{P.days_col(r)}"] = round(float(s[P.days_col(r)].median()), 1)
    row[f"mean_lat_{R_last}km"] = round(float(s[P.lat_col(R_last)].mean()), 2)
    row[f"mean_lon_{R_last}km"] = round(float(s[P.lon_col(R_last)].mean()), 2)
    rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(C.DATA_DIR / f"pathway_summary_k{BEST_K}_{LABEL_COL}.csv", index=False)
print(summary.to_string(index=False))

## Table 2 - Top-20 representative particles per label

The `N_REP` particles closest to their label's centroid in the space k-means
was fit in (standardized, weighted). Each row has `store` / `row` / `traj`, so
the real particle can be opened directly.

In [ ]:
N_REP = 20
Xw = scaler.transform(P.build_feature_matrix(labelled)) * W
diff = Xw - centers[labelled[LABEL_COL].to_numpy()]
labelled["dist_to_centroid"] = np.sqrt(np.einsum("ij,ij->i", diff, diff))

id_cols = [c for c in ("trajectory_id", "store", "row", "traj", "cluster_label",
                       "cluster_group", "status", "cross_time", "days_to_cross")
           if c in labelled.columns and c != LABEL_COL]
reps = (labelled.sort_values("dist_to_centroid").groupby(LABEL_COL, group_keys=False)
        .head(N_REP).sort_values([LABEL_COL, "dist_to_centroid"]).copy())
reps["name"] = reps[LABEL_COL].map(label_name)
reps["rank"] = reps.groupby(LABEL_COL).cumcount() + 1
reps = reps[[LABEL_COL, "name", "rank", "dist_to_centroid", *id_cols, *P.feature_columns()]]
out_csv = C.DATA_DIR / f"top{N_REP}_representatives_k{BEST_K}_{LABEL_COL}.csv"
reps.to_csv(out_csv, index=False)
print(f"wrote {len(reps):,} rows -> {out_csv}")
reps.head(N_REP)

In [ ]:
rep_pick = reps[["trajectory_id", "store", "row", "traj", LABEL_COL]] \
               .rename(columns={LABEL_COL: "lab"})
rep_pick = rep_pick.merge(labelled[["trajectory_id", "cross_obs"]], on="trajectory_id")
rep_paths = P.read_paths(rep_pick, n_jobs=N_IO_THREADS)

fig, axf = geo_axes_grid(n_lab)
for c in range(n_lab):
    ax = axf[c]
    for tid, k0 in rep_pick.loc[rep_pick.lab == c, ["trajectory_id", "cross_obs"]].itertuples(index=False):
        lo, la = rep_paths[int(tid)]
        m = ~(np.isnan(lo) | np.isnan(la)); m[:k0] = False      # after crossing only
        ax.plot(lo[m], la[m], "-", color=cmap(c), lw=.8, alpha=.7, transform=pc)
    ax.plot(np.r_[start_lon[c], disp_cent[c, 1::2]], np.r_[C.LAT_CROSS, disp_cent[c, 0::2]],
            "-o", color="k", lw=1.4, ms=2, zorder=5, transform=pc)
    land(ax)
    ax.set_title(f"{label_name(c)} (top {N_REP})", fontsize=9, color=cmap(c), fontweight="bold")
fig.suptitle(f"Top-{N_REP} representative particles per {LABEL_COL} (k={BEST_K}), "
             f"after crossing; black = centroid", y=1.0, fontsize=12)
fig.tight_layout()
fig.savefig(C.FIG_DIR / f"top{N_REP}_representatives_k{BEST_K}_{LABEL_COL}.png",
            dpi=130, bbox_inches="tight")
plt.show()

## Figure 6 - Animated: where each group leaves each circle
One frame per radius in `C.RADII_KM`.

In [ ]:
import matplotlib.animation as animation
from IPython.display import Image, display

GIF_FPS  = 1.2
GIF_NAME = C.FIG_DIR / f"radius_snapshots_k{BEST_K}_{LABEL_COL}.gif"

occ = {}
for r in C.RADII_KM:
    for c in range(n_lab):
        s = labelled.loc[labelled[LABEL_COL] == c, [P.lat_col(r), P.lon_col(r)]].dropna()
        h = np.histogram2d(s[P.lon_col(r)], s[P.lat_col(r)], bins=[lon_bins, lat_bins])[0]
        occ[(c, r)] = np.where(h.T > 0, 1.0, np.nan)

fig, axf = geo_axes_grid(n_lab, h=2.6)
meshes = []
for c in range(n_lab):
    land(axf[c])
    meshes.append(axf[c].pcolormesh(lon_bins, lat_bins, occ[(c, C.RADII_KM[0])],
                                    cmap=ListedColormap([cmap(c)]), vmin=0, vmax=1,
                                    alpha=.7, transform=pc, zorder=1))
    axf[c].set_title(label_name(c), fontsize=9, color=cmap(c), fontweight="bold")
sup = fig.suptitle("", y=1.0, fontsize=13)
fig.tight_layout()

def _frame(i):
    r = C.RADII_KM[i]
    for c in range(n_lab):
        meshes[c].set_array(occ[(c, r)].ravel())
    sup.set_text(f"Where each group leaves the {r} km circle (k={BEST_K})")
    return meshes + [sup]

anim = animation.FuncAnimation(fig, _frame, frames=len(C.RADII_KM), blit=False)
anim.save(GIF_NAME, writer=animation.PillowWriter(fps=GIF_FPS), dpi=90)
plt.close(fig)
display(Image(filename=GIF_NAME))